# fase_5 - script_afrida Migration

This notebook handles migration of database from old DB to new DB for fase 5.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: 4
Connected to future database: 4


## 2. Ambil Data dari DB Lama

In [3]:
import pandas as pd
import pickle

# Load mapping
with open('mapping_jadwal_detail.pkl', 'rb') as f:
    mapping_jadwal_detail = pickle.load(f)

print("=== Proses presensi_siswa ===")
query = "SELECT idpresensi_siswa, idjadwaldetil, idsiswa, waktu, status FROM presensi_siswa"
cursor_old.execute(query)
rows = cursor_old.fetchall()
df = pd.DataFrame(rows)

rename_dict = {
    'idpresensi_siswa': 'id_presensi_siswa_lama',
    'idjadwaldetil': 'id_jadwal_detail',
    'idsiswa': 'id_siswa',
    'waktu': 'waktu_presensi',
    'status': 'status_presensi'
}
df.rename(columns=rename_dict, inplace=True)

df['id_jadwal_detail'] = df['id_jadwal_detail'].map(mapping_jadwal_detail)
df.dropna(subset=['id_jadwal_detail'], inplace=True)
df['id_jadwal_detail'] = df['id_jadwal_detail'].astype('int64')

# Drop PK lama karena tidak sesuai tipe
df.drop(columns=['id_presensi_siswa_lama'], inplace=True)

df['waktu_presensi'] = pd.to_datetime(df['waktu_presensi'])
df['status_presensi'] = df['status_presensi'].astype('int8')

df_presensi_siswa = df.copy()
print(f"df_presensi_siswa shape: {df_presensi_siswa.shape}")

=== Proses presensi_siswa ===
df_presensi_siswa shape: (97762, 4)


In [4]:
display(df_presensi_siswa.head())

,id_jadwal_detail,id_siswa,waktu_presensi,status_presensi
0,87006,S0000362,2023-07-05 16:24:39,1
1,87006,S0000363,2023-07-05 16:24:40,1
2,86616,S0000378,2023-07-05 16:41:58,1
3,86616,S0000463,2023-07-05 16:42:00,1
4,86616,S0000464,2023-07-05 16:42:00,1


In [5]:
print("\n=== Proses catatan_siswa ===")

# 1. Ambil data dari DB lama
query_cs = "SELECT idcatatan_siswa, idjadwaldetil, idsiswa, catatan FROM catatan_siswa"
cursor_old.execute(query_cs)
rows_cs = cursor_old.fetchall()
df_cs = pd.DataFrame(rows_cs)

# 2. Simpan id_jadwal_detail_lama untuk lookup nanti (sebelum rename/mapping)
df_cs['id_jadwal_detail_lama'] = df_cs['idjadwaldetil']  # simpan original

# 3. Rename kolom
rename_cs = {
    'idcatatan_siswa': 'id_cs_lama',
    'idjadwaldetil': 'id_jadwal_detail',
    'idsiswa': 'id_siswa',
    'catatan': 'catatan_cs'
}
df_cs.rename(columns=rename_cs, inplace=True)
df_cs = df_cs[['id_cs_lama', 'id_jadwal_detail', 'id_siswa', 'catatan_cs', 'id_jadwal_detail_lama']].copy()

# 4. Mapping id_jadwal_detail
df_cs['id_jadwal_detail'] = df_cs['id_jadwal_detail'].map(mapping_jadwal_detail)
df_cs.dropna(subset=['id_jadwal_detail'], inplace=True)
df_cs['id_jadwal_detail'] = df_cs['id_jadwal_detail'].astype('int64')

# 5. Tambahkan id_jadwal dengan cara:
#    a. Ambil data jadwal_detail dari DB lama untuk mapping idjadwaldetil -> idjadwal
query_jd = "SELECT idjadwaldetil, idjadwal FROM jadwal_detil"
cursor_old.execute(query_jd)
rows_jd = cursor_old.fetchall()
# Buat dictionary: key = idjadwaldetil (varchar), value = idjadwal (varchar)
mapping_detil_to_jadwal = {row['idjadwaldetil']: row['idjadwal'] for row in rows_jd}

#    b. Load mapping_jadwal dari file pickle
with open('mapping_jadwal.pkl', 'rb') as f:
    mapping_jadwal = pickle.load(f)

#    c. Di df_cs, petakan id_jadwal_detail_lama -> id_jadwal_lama -> id_jadwal_baru
df_cs['id_jadwal_lama'] = df_cs['id_jadwal_detail_lama'].map(mapping_detil_to_jadwal)
df_cs['id_jadwal'] = df_cs['id_jadwal_lama'].map(mapping_jadwal)
# Hapus baris yang tidak punya id_jadwal valid
sebelum = len(df_cs)
df_cs.dropna(subset=['id_jadwal'], inplace=True)
sesudah = len(df_cs)
if sebelum > sesudah:
    print(f"⚠️ {sebelum - sesudah} baris dihapus karena id_jadwal tidak ditemukan di mapping.")
df_cs['id_jadwal'] = df_cs['id_jadwal'].astype('int64')

# 6. Generate id_cs baru (1..N) urut sesuai id_cs_lama
df_cs.sort_values('id_cs_lama', inplace=True)
df_cs.reset_index(drop=True, inplace=True)
old_ids_cs = df_cs['id_cs_lama'].copy()
df_cs['id_cs'] = range(1, len(df_cs)+1)
df_cs.drop(columns=['id_cs_lama'], inplace=True)

# 7. Simpan mapping untuk followup_cs
mapping_cs = dict(zip(old_ids_cs, df_cs['id_cs']))

# 8. Buang kolom bantu yang tidak diperlukan di hasil akhir
kolom_buang = ['id_jadwal_detail_lama', 'id_jadwal_lama']
df_cs.drop(columns=kolom_buang, inplace=True, errors='ignore')

# 9. Final dataframe
df_catatan_siswa = df_cs.copy()
print(f"df_catatan_siswa shape: {df_catatan_siswa.shape}")
print(f"Mapping cs siap: {len(mapping_cs)} pasang")
display(df_catatan_siswa.head())


=== Proses catatan_siswa ===
df_catatan_siswa shape: (1502, 5)
Mapping cs siap: 1502 pasang


,id_jadwal_detail,id_siswa,catatan_cs,id_jadwal,id_cs
0,87456,S0000229,She's good.,5499,1
1,87456,S0000230,He's good.,5499,2
2,87456,S0000156,He's good.,5499,3
3,87786,S0000273,Jojo didn't do the task before the zoom.,5510,4
4,87786,S0000169,Vian didn't do the task before the zoom,5510,5


In [6]:
# =================================================
# TAMBAHKAN KOLOM id_karyawan & tanggal (nullable)
# =================================================
# Pastikan df_catatan_siswa sudah ada dari proses sebelumnya

df_catatan_siswa['id_karyawan'] = None   # FK ke tabel karyawan, nullable
df_catatan_siswa['tanggal'] = None       # date, nullable

print("✅ Kolom id_karyawan & tanggal ditambahkan (NULL)")
display(df_catatan_siswa.head())
print("Kolom:", list(df_catatan_siswa.columns))

✅ Kolom id_karyawan & tanggal ditambahkan (NULL)


,id_jadwal_detail,id_siswa,catatan_cs,id_jadwal,id_cs,id_karyawan,tanggal
0,87456,S0000229,She's good.,5499,1,None,None
1,87456,S0000230,He's good.,5499,2,None,None
2,87456,S0000156,He's good.,5499,3,None,None
3,87786,S0000273,Jojo didn't do the task before the zoom.,5510,4,None,None
4,87786,S0000169,Vian didn't do the task before the zoom,5510,5,None,None


Kolom: ['id_jadwal_detail', 'id_siswa', 'catatan_cs', 'id_jadwal', 'id_cs', 'id_karyawan', 'tanggal']


In [7]:
display(df_catatan_siswa.head())

,id_jadwal_detail,id_siswa,catatan_cs,id_jadwal,id_cs,id_karyawan,tanggal
0,87456,S0000229,She's good.,5499,1,None,None
1,87456,S0000230,He's good.,5499,2,None,None
2,87456,S0000156,He's good.,5499,3,None,None
3,87786,S0000273,Jojo didn't do the task before the zoom.,5510,4,None,None
4,87786,S0000169,Vian didn't do the task before the zoom,5510,5,None,None


In [8]:
print("\nStruktur catatan_siswa (db_future):")
pd.read_sql("DESCRIBE catatan_siswa", db_future)


Struktur catatan_siswa (db_future):


,Field,Type,Null,Key,Default,Extra
0,id_cs,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_jadwal,bigint(20) unsigned,YES,MUL,None,
2,id_jadwal_detail,bigint(20) unsigned,YES,MUL,None,
3,id_siswa,bigint(20) unsigned,YES,MUL,None,
4,id_karyawan,bigint(20) unsigned,YES,,None,
5,tanggal,date,YES,,None,
6,catatan_cs,text,NO,,None,


In [9]:
# Cek info dataframe
print("=== Info DataFrame ===")
df_catatan_siswa.info()

print("\n=== Cek NULL per kolom ===")
print(df_catatan_siswa[['id_cs', 'id_jadwal_detail', 'catatan_cs']].isnull().sum())

print("\n=== Tipe data ===")
print(df_catatan_siswa[['id_cs', 'id_jadwal_detail', 'catatan_cs']].dtypes)

print("\n=== Contoh nilai unik id_cs (10 sampel) ===")
print(df_catatan_siswa['id_cs'].head(10))

print("\n=== Apakah id_cs bisa dikonversi ke numerik? ===")
# Cek apakah semua id_cs terdiri dari 'C' diikuti angka?
import re
pattern = r'^C\d+$'
mask = df_catatan_siswa['id_cs'].astype(str).str.match(pattern)
print(f"Jumlah yang sesuai format C<angka>: {mask.sum()} dari {len(df_catatan_siswa)}")
if not mask.all():
    print("Contoh yang tidak sesuai:")
    print(df_catatan_siswa.loc[~mask, 'id_cs'].head())

print("\n=== Cek anomali catatan_cs (misal terlalu panjang atau karakter aneh) ===")
print(f"Panjang maksimal catatan_cs: {df_catatan_siswa['catatan_cs'].str.len().max()}")
print(f"Ada null? {df_catatan_siswa['catatan_cs'].isnull().any()}")

=== Info DataFrame ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1502 entries, 0 to 1501
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_jadwal_detail  1502 non-null   int64 
 1   id_siswa          1502 non-null   object
 2   catatan_cs        1502 non-null   object
 3   id_jadwal         1502 non-null   int64 
 4   id_cs             1502 non-null   int64 
 5   id_karyawan       0 non-null      object
 6   tanggal           0 non-null      object
dtypes: int64(3), object(4)
memory usage: 82.3+ KB

=== Cek NULL per kolom ===
id_cs               0
id_jadwal_detail    0
catatan_cs          0
dtype: int64

=== Tipe data ===
id_cs                int64
id_jadwal_detail     int64
catatan_cs          object
dtype: object

=== Contoh nilai unik id_cs (10 sampel) ===
0     1
1     2
2     3
3     4
4     5
5     6
6     7
7     8
8     9
9    10
Name: id_cs, dtype: int64

=== Apakah id_cs bisa dikonversi 

In [10]:
import pandas as pd
import numpy as np

# ========== EKSTRAKSI DATA FOLLOWUP_CS ==========
print("="*60)
print("PROSES FOLLOWUP_CS (catatan_siswa_follow_up)")
print("="*60)

query = "SELECT * FROM catatan_siswa_follow_up"
cursor_old.execute(query)
rows = cursor_old.fetchall()
df_fu = pd.DataFrame(rows)
print(f"Jumlah baris awal: {len(df_fu)}")
print(f"Kolom asli: {list(df_fu.columns)}")

# ========== MAPPING KOLOM ==========
rename_map = {
    'idcatatan_siswa': 'id_cs',
    'tanggal': 'tanggal_followup',
    'idusers': 'id_user',
    'kesimpulan': 'kesimpulan_followup_cs',
    'status_follow': 'status_followup'
}
# idcs_follow_up tidak di-rename karena akan dihapus
df_fu.rename(columns=rename_map, inplace=True)

# Kolom target: tanpa id_followup_cs
target_cols = ['id_cs', 'tanggal_followup', 'id_user', 'kesimpulan_followup_cs', 'status_followup']
df_fu = df_fu[[col for col in target_cols if col in df_fu.columns]]
print(f"Setelah mapping kolom: {list(df_fu.columns)}")

# ========== TERAPKAN MAPPING ID_CS ==========
# Pastikan mapping_cs sudah ada
if 'mapping_cs' not in dir():
    print("ERROR: mapping_cs belum tersedia. Jalankan proses catatan_siswa dulu.")
else:
    df_fu['id_cs'] = df_fu['id_cs'].map(mapping_cs)
    before = len(df_fu)
    df_fu.dropna(subset=['id_cs'], inplace=True)
    after = len(df_fu)
    if before != after:
        print(f"⚠️ {before - after} baris di-drop karena id_cs tidak valid (tidak ada di mapping)")

# ========== CEK ANOMALI PER KOLOM ==========
print("\n" + "="*60)
print("CEK ANOMALI FOLLOWUP_CS")
print("="*60)

# 1. Tipe data
print("\n[1] TIPE DATA SAAT INI:")
print(df_fu.dtypes)

# 2. NULL counts
print("\n[2] NULL COUNTS:")
null_counts = df_fu.isnull().sum()
print(null_counts)
if null_counts.sum() > 0:
    print("⚠️ Ada NULL di kolom:", null_counts[null_counts > 0].index.tolist())

# 3. id_cs: setelah mapping seharusnya tidak ada NULL, integer
print("\n[3] KOLOM id_cs:")
print(f"   Unique values: {df_fu['id_cs'].nunique()}")
print(f"   Duplicated: {df_fu['id_cs'].duplicated().sum()}")
print(f"   Min: {df_fu['id_cs'].min()}, Max: {df_fu['id_cs'].max()}")
# Cek apakah semua id_cs ada di tabel catatan_siswa (opsional, karena mapping berasal dari sana)
if 'df_catatan_siswa' in dir():
    valid_ids = set(df_catatan_siswa['id_cs'].unique())
    invalid = set(df_fu['id_cs'].unique()) - valid_ids
    if invalid:
        print(f"   ⚠️ {len(invalid)} id_cs tidak ada di catatan_siswa: {list(invalid)[:5]}")

# 4. tanggal_followup: konversi ke datetime dan cek
print("\n[4] KOLOM tanggal_followup:")
# Coba konversi
df_fu['tanggal_followup'] = pd.to_datetime(df_fu['tanggal_followup'], errors='coerce')
failed_conv = df_fu['tanggal_followup'].isnull().sum()
if failed_conv > 0:
    print(f"   ⚠️ {failed_conv} baris gagal konversi ke datetime (dijadi NaT). Contoh:")
    contoh_gagal = df_fu[df_fu['tanggal_followup'].isnull()]['tanggal_followup'].head(3).tolist()
    print(f"      {contoh_gagal}")
else:
    print("   ✅ Semua berhasil dikonversi ke datetime.")
print(f"   Rentang tanggal: {df_fu['tanggal_followup'].min()} s.d. {df_fu['tanggal_followup'].max()}")

# 5. id_user: cek tipe, null, nilai unik, panjang
print("\n[5] KOLOM id_user (string):")
print(f"   Tipe: {df_fu['id_user'].dtype}")
print(f"   NULL: {df_fu['id_user'].isnull().sum()}")
print(f"   Unique values: {df_fu['id_user'].nunique()}")
print(f"   Contoh nilai: {df_fu['id_user'].dropna().unique()[:5].tolist()}")
# Cek panjang maksimal
if df_fu['id_user'].dtype == 'object':
    max_len = df_fu['id_user'].dropna().astype(str).str.len().max()
    print(f"   Panjang maksimal: {max_len}")
# Opsional: cek apakah id_user ada di tabel users DB baru (jika perlu)
# cursor_new.execute("SELECT id_user FROM users")
# existing_users = {row['id_user'] for row in cursor_new.fetchall()}
# missing_users = set(df_fu['id_user'].dropna().unique()) - existing_users
# if missing_users: print(f"   ⚠️ {len(missing_users)} id_user tidak ditemukan di users: {list(missing_users)[:5]}")

# 6. kesimpulan_followup_cs: teks, cek null dan panjang
print("\n[6] KOLOM kesimpulan_followup_cs (text):")
print(f"   NULL: {df_fu['kesimpulan_followup_cs'].isnull().sum()}")
print(f"   Unique values: {df_fu['kesimpulan_followup_cs'].nunique()}")
if df_fu['kesimpulan_followup_cs'].dtype == 'object':
    max_len = df_fu['kesimpulan_followup_cs'].dropna().astype(str).str.len().max()
    print(f"   Panjang maksimal: {max_len}")
    # Contoh nilai (jika tidak terlalu banyak)
    if df_fu['kesimpulan_followup_cs'].nunique() <= 20:
        print(f"   Nilai unik: {df_fu['kesimpulan_followup_cs'].unique().tolist()}")
    else:
        print(f"   Contoh nilai: {df_fu['kesimpulan_followup_cs'].dropna().iloc[0][:100]}...")

# 7. status_followup: cek nilai enum
print("\n[7] KOLOM status_followup (enum):")
print(f"   NULL: {df_fu['status_followup'].isnull().sum()}")
unique_status = df_fu['status_followup'].dropna().unique()
print(f"   Nilai unik: {unique_status.tolist()}")
expected_enum = ['NEED FURTHER OBSERVATION', 'CASE CLOSED']
if set(unique_status).issubset(expected_enum):
    print("   ✅ Semua nilai sesuai dengan enum yang diharapkan.")
else:
    unexpected = set(unique_status) - set(expected_enum)
    print(f"   ⚠️ Nilai tidak sesuai enum: {unexpected}")

# 8. Duplikat baris (keseluruhan)
print("\n[8] DUPLIKAT BARIS (keseluruhan):")
duplicated_rows = df_fu.duplicated().sum()
print(f"   {duplicated_rows} baris duplikat sempurna.")

# 9. Statistik umum
print("\n[9] STATISTIK UMUM:")
print(f"   Jumlah baris setelah drop invalid id_cs: {len(df_fu)}")
print(f"   Memory usage: {df_fu.memory_usage(deep=True).sum() / 1024:.2f} KB")

print("\n✅ Pemeriksaan selesai. Dataframe siap di variabel 'df_followup_cs'")
# Simpan ke variabel
df_followup_cs = df_fu.copy()

PROSES FOLLOWUP_CS (catatan_siswa_follow_up)
Jumlah baris awal: 22
Kolom asli: ['idcs_follow_up', 'idcatatan_siswa', 'tanggal', 'idusers', 'kesimpulan', 'status_follow']
Setelah mapping kolom: ['id_cs', 'tanggal_followup', 'id_user', 'kesimpulan_followup_cs', 'status_followup']

CEK ANOMALI FOLLOWUP_CS

[1] TIPE DATA SAAT INI:
id_cs                      int64
tanggal_followup          object
id_user                   object
kesimpulan_followup_cs    object
status_followup           object
dtype: object

[2] NULL COUNTS:
id_cs                     0
tanggal_followup          0
id_user                   0
kesimpulan_followup_cs    0
status_followup           0
dtype: int64

[3] KOLOM id_cs:
   Unique values: 22
   Duplicated: 0
   Min: 12, Max: 480

[4] KOLOM tanggal_followup:
   ✅ Semua berhasil dikonversi ke datetime.
   Rentang tanggal: 2023-07-14 00:00:00 s.d. 2023-09-22 00:00:00

[5] KOLOM id_user (string):
   Tipe: object
   NULL: 0
   Unique values: 3
   Contoh nilai: ['U00026', 'U

In [11]:
display(df_followup_cs.head())

,id_cs,tanggal_followup,id_user,kesimpulan_followup_cs,status_followup
0,12,2023-07-14,U00026,"Okay, bantu FU - Qorin",NEED FURTHER OBSERVATION
1,56,2023-07-14,U00011,"done keluarkan LV dan WAG yah, Sarah akan kemb...",NEED FURTHER OBSERVATION
2,59,2023-07-14,U00011,"Rehan blm bayar SPP, sudah di japri Daniar blm...",NEED FURTHER OBSERVATION
3,63,2023-07-20,U00011,"sudah masuk, dan mama sudah bersedia ditagih S...",CASE CLOSED
4,132,2023-07-28,U00011,"(CS28)\r\nMiss Daniar , ini jika nanti Miss Ri...",NEED FURTHER OBSERVATION


In [12]:
print("\nStruktur FOLLOWUP_CS (db_future):")
pd.read_sql("DESCRIBE followup_cs", db_future)


Struktur FOLLOWUP_CS (db_future):


,Field,Type,Null,Key,Default,Extra
0,id_followup_cs,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_cs,bigint(20) unsigned,YES,MUL,None,
2,tanggal_followup,timestamp,NO,,current_timestamp(),
3,id_user,varchar(20),YES,MUL,None,
4,kesimpulan_followup_cs,text,NO,,None,
5,status_followup,"enum('NEED FURTHER OBSERVATION','CASE CLOSED')",NO,,NEED FURTHER OBSERVATION,


In [13]:
import pickle

# Asumsikan df_presensi_siswa, df_catatan_siswa, df_followup_cs sudah ada di memori
# Jika tidak, kamu harus load atau proses ulang.

data_to_save = {
    'presensi_siswa': df_presensi_siswa,
    'catatan_siswa': df_catatan_siswa,
    'followup_cs': df_followup_cs
}

with open('fase_5_afrida.pkl', 'wb') as f:
    pickle.dump(data_to_save, f)

print("✅ Semua dataframe disimpan ke fase_5_afrida.pkl")
print("Keys:", data_to_save.keys())

✅ Semua dataframe disimpan ke fase_5_afrida.pkl
Keys: dict_keys(['presensi_siswa', 'catatan_siswa', 'followup_cs'])


## 3. Transform Data (jika diperlukan)

## 4. Insert ke DB Baru

## 5. Verifikasi Data

## 6. Return Hasil Migrasi untuk migrate_db.py

## 7. Close Connection